# 🔧 Data Preprocessing & Feature Engineering

**Project**: Carbon Footprint AI Microservice  
**Step**: Phase 3 - Data Preprocessing

---

## 📋 Objectives

This notebook focuses on preparing the ASHRAE dataset for Machine Learning models:

1. **Data Cleaning**: Handle missing values and outliers
2. **Data Merging**: Combine building, weather, and meter data
3. **Feature Engineering**: Create temporal and building-specific features
4. **Data Transformation**: Encode categorical variables and scale numerical ones
5. **Data Splitting**: Prepare train/val/test sets

---

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Settings
pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings('ignore')

# Paths
RAW_DATA_DIR = '../../data/raw'
PROCESSED_DATA_DIR = '../../data/processed'

# Ensure processed directory exists
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

## 1️⃣ Load Data

In [ ]:
def load_data():
    print("Loading data...")
    building_df = pd.read_csv(os.path.join(RAW_DATA_DIR, 'building_metadata.csv'))
    weather_train_df = pd.read_csv(os.path.join(RAW_DATA_DIR, 'weather_train.csv'))
    # Load a subset of train data for demonstration/memory saving if needed
    # For full training, remove nrows
    train_df = pd.read_csv(os.path.join(RAW_DATA_DIR, 'train.csv')) #, nrows=1000000)
    
    print(f"Building data shape: {building_df.shape}")
    print(f"Weather train shape: {weather_train_df.shape}")
    print(f"Train data shape: {train_df.shape}")
    return building_df, weather_train_df, train_df

building_df, weather_train_df, train_df = load_data()

## 2️⃣ Data Cleaning & Memory Optimization

The dataset is large, so we'll optimize memory usage by downcasting types.

In [ ]:
def reduce_mem_usage(df):
    """ Iterate through all the columns of a dataframe and modify the data type
        to reduce memory usage.        
    """
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object and col_type.name != 'category' and 'datetime' not in col_type.name:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    
    return df

building_df = reduce_mem_usage(building_df)
weather_train_df = reduce_mem_usage(weather_train_df)
train_df = reduce_mem_usage(train_df)

### 2.1 Handle Missing Values

- **Weather Data**: Interpolate missing values (time-series nature).
- **Building Data**: Fill `year_built` and `floor_count` with median or drop if too many missing.

In [ ]:
# Convert timestamps
weather_train_df['timestamp'] = pd.to_datetime(weather_train_df['timestamp'])
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])

# Weather Interpolation (per site)
weather_train_df = weather_train_df.groupby('site_id').apply(lambda group: group.interpolate(limit_direction='both'))

# Fill remaining weather NaNs with global mean
for col in weather_train_df.columns:
    if weather_train_df[col].isnull().any():
        weather_train_df[col] = weather_train_df[col].fillna(weather_train_df[col].mean())

# Building Metadata
# floor_count and year_built have many missing values. 
# We can fill with median or create a 'is_missing' flag, or just drop for baseline.
# For now, let's fill with median of the primary_use group
building_df['year_built'] = building_df.groupby('primary_use')['year_built'].transform(lambda x: x.fillna(x.median()))
building_df['floor_count'] = building_df.groupby('primary_use')['floor_count'].transform(lambda x: x.fillna(x.median()))

# Fill remaining with global median
building_df['year_built'] = building_df['year_built'].fillna(building_df['year_built'].median())
building_df['floor_count'] = building_df['floor_count'].fillna(building_df['floor_count'].median())

print("Missing values handled.")

## 3️⃣ Feature Engineering

In [ ]:
# Merge Data
print("Merging data...")
train_df = train_df.merge(building_df, on='building_id', how='left')
train_df = train_df.merge(weather_train_df, on=['site_id', 'timestamp'], how='left')

print(f"Merged shape: {train_df.shape}")

In [ ]:
# Temporal Features
train_df['hour'] = train_df['timestamp'].dt.hour
train_df['day'] = train_df['timestamp'].dt.day
train_df['weekday'] = train_df['timestamp'].dt.weekday
train_df['month'] = train_df['timestamp'].dt.month
train_df['is_weekend'] = train_df['weekday'].apply(lambda x: 1 if x >= 5 else 0)

# Building Features
# Log transform square_feet because it's skewed
train_df['log_square_feet'] = np.log1p(train_df['square_feet'])

# Building Age
current_year = 2017 # Dataset is from 2016/2017
train_df['building_age'] = current_year - train_df['year_built']

# Target Transformation
# Meter reading is highly skewed. We'll use log1p for training.
train_df['log_meter_reading'] = np.log1p(train_df['meter_reading'])

print("Features created.")

## 4️⃣ Encoding & Scaling

In [ ]:
# Label Encoding for Categorical Variables
le = LabelEncoder()
train_df['primary_use'] = le.fit_transform(train_df['primary_use'])

# Save the encoder classes for later use (e.g. in API)
np.save(os.path.join(PROCESSED_DATA_DIR, 'primary_use_classes.npy'), le.classes_)

print("Categorical variables encoded.")

## 5️⃣ Save Processed Data

We'll save the processed dataframe to Parquet format for faster loading and smaller size.

In [ ]:
# Drop timestamp (we extracted features) and original meter_reading (we have log)
# But keep timestamp for splitting or visualization if needed. 
# Let's drop 'meter_reading' to save space, we can reverse log1p later.

cols_to_drop = ['year_built', 'square_feet'] # We have age and log_sq_ft
train_df.drop(columns=cols_to_drop, inplace=True)

output_path = os.path.join(PROCESSED_DATA_DIR, 'train_processed.parquet')
print(f"Saving to {output_path}...")
train_df.to_parquet(output_path, index=False)

print("✅ Data preprocessing complete!")